# 🌸 Iris Dataset — Data Models Training & Testing
---

**Dataset :** Iris Flower Dataset · 150 samples · 3 species · 4 features  
**Libraries:** Seaborn · Matplotlib · Scikit-learn · Pandas · NumPy  
**Models Covered:**
| # | Model | Type |
|---|---|---|
| 1 | Logistic Regression | Linear Classifier |
| 2 | K-Nearest Neighbors (KNN) | Instance-based |
| 3 | Decision Tree | Tree-based |
| 4 | Random Forest | Ensemble |
| 5 | Support Vector Machine (SVM) | Margin-based |
---

* ##  Importing Libraries

We import visualization libraries (**Seaborn**, **Matplotlib**), data-processing libraries (**Pandas**, **NumPy**), and Scikit-learn modules for model building, evaluation, and preprocessing.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.metrics         import (accuracy_score, classification_report,
                                     confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model    import LogisticRegression
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.tree            import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from pathlib                 import Path

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13,
                     "axes.labelsize": 11, "legend.fontsize": 10})

---
* ##  Loading & Preparing the Dataset

We load the Iris CSV, drop the index column, rename features for cleanliness, and strip the `Iris-` prefix from species labels. We then encode the target column into integers (required by most sklearn models).

 Species | Description |
|---|---|
| *Iris-setosa* | Easily separable — small petals |
| *Iris-versicolor* | Overlaps slightly with virginica |
| *Iris-virginica* | Largest petals among the three |

Each sample records **four measurements** (in centimetres):
- Sepal Length & Sepal Width
- Petal Length & Petal Width



In [ ]:
BASE_DIR = Path.cwd().parent
file_path = BASE_DIR / 'data' / 'Iris.csv'

df = pd.read_csv(file_path)
df.drop(columns=['Id'], inplace=True)
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df['species'] = df['species'].str.replace('Iris-', '', regex=False)


le = LabelEncoder()
df['species_enc'] = le.fit_transform(df['species'])   
class_names = le.classes_

print(f"Shape          : {df.shape}")
print(f"Species labels : {class_names.tolist()} → {[0,1,2]}")
df.head(8)

---
* ## Train / Test Split & Feature Scaling

We split the data **80 % training / 20 % test** with stratification so each species is equally represented in both sets. Feature scaling (StandardScaler) is applied for models sensitive to feature magnitude (Logistic Regression, KNN, SVM).


In [ ]:
X = df[['sepal_length','sepal_width','petal_length','petal_width']].values
y = df['species_enc'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples : {X_train.shape[0]}  (per class: {dict(zip(*np.unique(y_train, return_counts=True)))})")
print(f"Test samples     : {X_test.shape[0]}   (per class: {dict(zip(*np.unique(y_test,  return_counts=True)))})")

---
* ## Reusable Helper Functions

We define a helper to **plot a confusion matrix** and another to **visualise decision boundaries** using the two most discriminative features (petal_length & petal_width). These will be reused for all five models.


In [ ]:
def plot_confusion(y_true, y_pred, model_name, ax=None):
    """Plot a labelled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    show = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"Confusion Matrix — {model_name}", fontweight='bold')
    if show:
        plt.tight_layout(); plt.show()


def plot_decision_boundary(clf, X_sc, y, model_name, feat_idx=(2, 3)):
    """2-D decision boundary using two scaled features."""
    f0, f1 = feat_idx
    feat_names = ['sepal_length','sepal_width','petal_length','petal_width']
    X2 = X_sc[:, [f0, f1]]

    x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
    y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))

    grid = np.zeros((xx.ravel().shape[0], X_sc.shape[1]))
    grid[:, f0] = xx.ravel()
    grid[:, f1] = yy.ravel()

    Z = clf.predict(grid).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='Set2')
    scatter = ax.scatter(X2[:, 0], X2[:, 1], c=y,
                         cmap='Set2', edgecolors='k', s=60, zorder=3)
    ax.set_xlabel(feat_names[f0] + " (scaled)")
    ax.set_ylabel(feat_names[f1] + " (scaled)")
    ax.set_title(f"Decision Boundary — {model_name}", fontweight='bold')
    legend_labels = [plt.Line2D([0],[0], marker='o', color='w',
                                markerfacecolor=sns.color_palette('Set2')[i],
                                markersize=9, label=class_names[i]) for i in range(3)]
    ax.legend(handles=legend_labels, title='Species')
    plt.tight_layout(); plt.show()

---
* ## Model 1 — Logistic Regression

**Logistic Regression** is a linear classifier that models the probability of each class using the logistic (sigmoid) function. Despite the word "regression", it is used for **classification**. It works best when classes are linearly separable.

**How it works:**
- Fits a hyperplane to separate classes.
- Outputs probabilities via the softmax function (multi-class via OvR or multinomial).
- Regularisation parameter `C` controls overfitting — smaller `C` = stronger regularisation.


In [ ]:
lr = LogisticRegression(C=1.0, max_iter=200, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression — Test Accuracy: {acc_lr*100:.2f}%")
print()
print(classification_report(y_test, y_pred_lr, target_names=class_names))

The classification report shows **precision, recall, and F1-score** for each species:
- **Precision**: Of all predicted as class X, how many were actually X?
- **Recall**: Of all actual class X, how many did we correctly predict?
- **F1-score**: Harmonic mean of precision and recall.


In [ ]:
plot_confusion(y_test, y_pred_lr, "Logistic Regression")

In [ ]:
plot_decision_boundary(lr, X_test_sc, y_test, "Logistic Regression")

The decision boundary is **linear** — straight lines separate the three regions. Logistic Regression works well here because petal features are nearly linearly separable. Setosa (top-left) is always cleanly separated.

In [ ]:
cv_lr = cross_val_score(lr, scaler.transform(X), y, cv=StratifiedKFold(5), scoring='accuracy')
print(f"5-Fold CV Accuracy : {cv_lr.mean()*100:.2f}% ± {cv_lr.std()*100:.2f}%")
print(f"Per-fold scores    : {[f'{s*100:.1f}%' for s in cv_lr]}")

---
* ## Model 2 — K-Nearest Neighbors (KNN)

**KNN** is a non-parametric, instance-based algorithm. It classifies a new point by looking at the **K closest training samples** and taking a majority vote of their labels.

**Key hyperparameter — K:**  
- Small K (e.g. K=1) → very flexible, may overfit.  
- Large K → smoother boundaries, may underfit.  
- Choosing optimal K via accuracy vs K plot is standard practice.


In [ ]:
k_range = range(1, 21)
k_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                             scaler.transform(X), y,
                             cv=StratifiedKFold(5), scoring='accuracy').mean()
            for k in k_range]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_range, k_scores, 'o-', color='steelblue', linewidth=2, markersize=7)
ax.axvline(x=k_scores.index(max(k_scores))+1, color='red', linestyle='--', label=f"Best K={k_scores.index(max(k_scores))+1}")
ax.set_xlabel("Number of Neighbors (K)")
ax.set_ylabel("CV Accuracy")
ax.set_title("KNN — Cross-Validation Accuracy vs. K", fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

best_k = k_scores.index(max(k_scores)) + 1
print(f"Best K = {best_k}  (CV Accuracy = {max(k_scores)*100:.2f}%)")

The plot shows how accuracy changes as K increases. We select the **K with the highest cross-validation accuracy** as our optimal hyperparameter.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)

acc_knn = accuracy_score(y_test, y_pred_knn)
print(f"KNN (K={best_k}) — Test Accuracy: {acc_knn*100:.2f}%")
print()
print(classification_report(y_test, y_pred_knn, target_names=class_names))

In [ ]:
plot_confusion(y_test, y_pred_knn, f"KNN (K={best_k})")

In [ ]:
plot_decision_boundary(knn, X_test_sc, y_test, f"KNN (K={best_k})")

KNN's decision boundary is **non-linear** — it forms irregular, locally adaptive regions based on the training data distribution. This allows it to capture more complex patterns than Logistic Regression.

---
* ## Model 3 — Decision Tree Classifier

A **Decision Tree** splits the data recursively using the feature and threshold that best separates the classes at each step. It is highly interpretable — we can visualise the exact rules the model uses to classify.

**Split criterion:** Gini impurity (default) — measures how often a randomly chosen sample would be misclassified.  
**Max depth** controls overfitting: deeper trees memorise training data; shallower trees generalise better.


In [ ]:

dt = DecisionTreeClassifier(max_depth=4, random_state=42, criterion='gini')
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
print(f"Decision Tree — Test Accuracy: {acc_dt*100:.2f}%")
print()
print(classification_report(y_test, y_pred_dt, target_names=class_names))


In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(dt,
          feature_names=['sepal_length','sepal_width','petal_length','petal_width'],
          class_names=class_names,
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("Decision Tree — Full Tree Visualisation (max_depth=4)", fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

The tree is fully interpretable. **Each node shows:**
- The splitting feature and threshold (e.g. `petal_length ≤ 2.45`)
- Gini impurity of the node
- Number of samples reaching that node
- Class distribution & predicted class (colour = dominant class)

Reading from root: the first split on **petal_length ≤ 2.45** immediately separates all setosa samples.


print(export_text(dt, feature_names=['sepal_length','sepal_width','petal_length','petal_width']))

In [ ]:
plot_confusion(y_test, y_pred_dt, "Decision Tree")

In [ ]:
feat_imp = pd.Series(dt.feature_importances_,
                     index=['sepal_length','sepal_width','petal_length','petal_width']).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
feat_imp.plot(kind='barh', color=sns.color_palette('Set2', 4), ax=ax)
ax.set_title("Decision Tree — Feature Importances", fontweight='bold')
ax.set_xlabel("Gini Importance Score")
plt.tight_layout(); plt.show()

The feature importance chart confirms what the pair plot suggested — **petal features dominate**, with `petal_length` being the single most important feature for classification.

In [ ]:
depths = range(1, 11)
train_accs = [DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
              .score(X_train, y_train) for d in depths]
test_accs  = [DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
              .score(X_test, y_test) for d in depths]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, [a*100 for a in train_accs], 'o-', label='Train Accuracy', color='steelblue', linewidth=2)
ax.plot(depths, [a*100 for a in test_accs],  's-', label='Test Accuracy',  color='tomato',    linewidth=2)
ax.axvline(x=4, color='grey', linestyle='--', alpha=0.7, label='Chosen depth=4')
ax.set_xlabel("Tree Depth"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Decision Tree — Train vs Test Accuracy by Depth", fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

The depth vs accuracy curve illustrates the **bias–variance tradeoff**: very shallow trees underfit (high bias), while very deep trees overfit (high variance — training accuracy = 100% but test accuracy plateaus). Depth 4 is a good sweet spot.

---
* ## Model 4 — Random Forest Classifier

**Random Forest** is an **ensemble** method that builds many decision trees on random subsets of the data (bagging) and features, then aggregates their predictions by majority vote.

**Why it improves on a single Decision Tree:**
- Reduces variance — individual trees overfit, but their average does not.
- More robust to noisy features and outliers.
- Provides reliable feature importance estimates.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=4,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest — Test Accuracy: {acc_rf*100:.2f}%")
print()
print(classification_report(y_test, y_pred_rf, target_names=class_names))

In [ ]:
plot_confusion(y_test, y_pred_rf, "Random Forest")

In [ ]:
rf_imp = pd.Series(rf.feature_importances_,
                   index=['sepal_length','sepal_width','petal_length','petal_width']).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
rf_imp.plot(kind='barh', color=sns.color_palette('Set2', 4), ax=ax)
ax.set_title("Random Forest — Feature Importances (mean over 100 trees)", fontweight='bold')
ax.set_xlabel("Mean Gini Importance")
plt.tight_layout(); plt.show()

Random Forest's feature importance is averaged across all 100 trees, making it more stable than a single decision tree's estimate. `petal_length` again dominates, followed by `petal_width`.


In [ ]:
rf_oob = RandomForestClassifier(n_estimators=200, oob_score=True,
                                 random_state=42, n_jobs=-1)
rf_oob.fit(X_train, y_train)
oob_scores = []
for n in range(10, 201, 10):
    rf_tmp = RandomForestClassifier(n_estimators=n, oob_score=True,
                                    random_state=42, n_jobs=-1)
    rf_tmp.fit(X_train, y_train)
    oob_scores.append(rf_tmp.oob_score_ * 100)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(10, 201, 10), oob_scores, 'o-', color='darkorchid', linewidth=2, markersize=5)
ax.set_xlabel("Number of Trees (n_estimators)")
ax.set_ylabel("OOB Accuracy (%)")
ax.set_title("Random Forest — OOB Accuracy vs. Number of Trees", fontweight='bold')
plt.tight_layout(); plt.show()


The **Out-of-Bag (OOB) accuracy** is a free cross-validation estimate — samples not used in a tree's bootstrap are used to evaluate it. The curve stabilises after roughly 50 trees, confirming that 100 trees is more than sufficient.


---
* ## Model 5 — Support Vector Machine (SVM)

**SVM** finds the **maximum-margin hyperplane** that best separates the classes. Points closest to the boundary are called **support vectors**. With the **RBF kernel**, SVM maps the data to a higher-dimensional space, allowing non-linear boundaries.

**Key hyperparameters:**
- `C`: Regularisation — larger C = narrower margin, fits training data more tightly.
- `gamma`: Kernel bandwidth — larger gamma = more locally fitted boundary.


svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_sc, y_train)
y_pred_svm = svm.predict(X_test_sc)

acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"SVM (RBF) — Test Accuracy: {acc_svm*100:.2f}%")
print()
print(classification_report(y_test, y_pred_svm, target_names=class_names))


In [ ]:
plot_confusion(y_test, y_pred_svm, "SVM (RBF Kernel)")


In [ ]:
plot_decision_boundary(svm, X_test_sc, y_test, "SVM (RBF Kernel)")

The SVM boundary is **curved / non-linear** (due to the RBF kernel), allowing it to wrap around more complex class structures. This often improves performance on borderline samples between versicolor and virginica.


In [ ]:
C_vals = [0.01, 0.1, 0.5, 1, 5, 10, 50, 100]
c_cv_scores = [cross_val_score(SVC(kernel='rbf', C=c, gamma='scale'),
                                scaler.transform(X), y,
                                cv=StratifiedKFold(5)).mean() * 100
               for c in C_vals]

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(C_vals, c_cv_scores, 's-', color='tomato', linewidth=2, markersize=8)
ax.set_xlabel("C (log scale)")
ax.set_ylabel("CV Accuracy (%)")
ax.set_title("SVM — Cross-Validation Accuracy vs. Regularisation C", fontweight='bold')
plt.tight_layout(); plt.show()

The accuracy remains high for a wide range of C values, showing that SVM is robust on this dataset. Very small C (strong regularisation) degrades performance; very large C can overfit but matters less here due to the clean data.


---
* ## All Confusion Matrices — Side by Side

Comparing all five models' confusion matrices in one view makes it easy to spot which species each model struggles with most.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
models_info = [
    ("Logistic Reg.", y_pred_lr),
    (f"KNN (K={best_k})",  y_pred_knn),
    ("Decision Tree",  y_pred_dt),
    ("Random Forest",  y_pred_rf),
    ("SVM (RBF)",      y_pred_svm),
]
for ax, (name, preds) in zip(axes, models_info):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names, cbar=False,
                annot_kws={'size':13,'weight':'bold'})
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual" if ax == axes[0] else "")
    ax.tick_params(axis='x', rotation=30)

plt.suptitle("Confusion Matrices — All 5 Models", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


A perfect confusion matrix has non-zero values only on the diagonal (zero off-diagonal = zero misclassifications). The off-diagonal cells represent errors — almost always between **versicolor and virginica**, confirming these are the hardest two species to separate.

---
* ## Cross-Validation Accuracy Comparison

A single train/test split can give misleading results depending on which samples happen to fall in the test set. **Stratified 5-fold cross-validation** provides a more reliable estimate of generalisation performance.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_sc_all = scaler.transform(X)

results = {}
models_cv = {
    "Logistic Regression" : LogisticRegression(C=1.0, max_iter=200, random_state=42),
    f"KNN (K={best_k})"   : KNeighborsClassifier(n_neighbors=best_k),
    "Decision Tree"        : DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest"        : RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1),
    "SVM (RBF)"            : SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
}

for name, model in models_cv.items():
    # Decision Tree & RF don't need scaling, others do
    Xuse = X if name in ("Decision Tree","Random Forest") else X_sc_all
    scores = cross_val_score(model, Xuse, y, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f"{name:<22} — Mean: {scores.mean()*100:.2f}%  Std: {scores.std()*100:.2f}%")


In [ ]:
names  = list(results.keys())
means  = [results[n].mean()*100 for n in names]
stds   = [results[n].std()*100  for n in names]
colors = sns.color_palette("Set2", len(names))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, means, yerr=stds, capsize=6, color=colors,
              edgecolor='black', linewidth=0.8)
ax.set_ylim(85, 105)
ax.set_ylabel("CV Accuracy (%)")
ax.set_title("5-Fold Cross-Validation Accuracy — All Models", fontweight='bold')
ax.set_xticklabels(names, rotation=15, ha='right')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{mean:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
cv_df = pd.DataFrame({n: results[n]*100 for n in names})
fig, ax = plt.subplots(figsize=(10, 5))
cv_df.boxplot(ax=ax, patch_artist=True,
              boxprops=dict(facecolor='lightsteelblue', color='steelblue'),
              medianprops=dict(color='red', linewidth=2))
ax.set_ylabel("Accuracy per Fold (%)")
ax.set_title("Per-Fold Accuracy Distribution — 5-Fold CV", fontweight='bold')
ax.set_xticklabels(names, rotation=15, ha='right')
plt.tight_layout(); plt.show()

The box plots reveal not just the **average accuracy** but also **consistency across folds**. A model with a high mean but large spread is less reliable than one with a slightly lower mean but tight spread.


---
* ## Final Model Comparison Summary Table


In [ ]:
test_preds = {
    "Logistic Regression" : y_pred_lr,
    f"KNN (K={best_k})"   : y_pred_knn,
    "Decision Tree"        : y_pred_dt,
    "Random Forest"        : y_pred_rf,
    "SVM (RBF)"            : y_pred_svm,
}

rows = []
for name in names:
    preds = test_preds[name]
    rows.append({
        "Model"              : name,
        "Test Accuracy (%)"  : round(accuracy_score(y_test, preds)*100, 2),
        "CV Mean (%)"        : round(results[name].mean()*100, 2),
        "CV Std (%)"         : round(results[name].std()*100, 2),
        "Interpretable"      : "Yes" if name == "Decision Tree" else ("Partial" if name == "Logistic Regression" else "No"),
        "Needs Scaling"      : "No" if name in ("Decision Tree","Random Forest") else "Yes",
        "Boundary Type"      : {"Logistic Regression":"Linear",
                                f"KNN (K={best_k})":"Non-linear (local)",
                                "Decision Tree":"Axis-aligned rectangles",
                                "Random Forest":"Axis-aligned (ensemble)",
                                "SVM (RBF)":"Non-linear (RBF kernel)"}[name],
    })

comp_df = pd.DataFrame(rows).set_index("Model")
comp_df


---

* ## Conclusions

### Dataset
- The Iris dataset is **perfectly balanced** (50 samples per species) with four continuous features.
- **Petal length and petal width** are the most discriminative features — setosa is always cleanly separable, while versicolor and virginica overlap slightly.
- Petal length and petal width are highly correlated (r = 0.96), so one is nearly redundant.

### Models
| Model | Strength | Limitation |
|---|---|---|
| **Logistic Regression** | Fast, interpretable, great baseline | Assumes linearity; may underfit complex boundaries |
| **KNN** | No training phase, captures local patterns | Slow at prediction; sensitive to feature scale and noisy data |
| **Decision Tree** | Fully interpretable, handles non-linearity | Prone to overfitting; unstable (high variance) |
| **Random Forest** | Best generalisation, robust to noise | Less interpretable; slower to train |
| **SVM (RBF)** | Strong performance with RBF kernel, handles non-linearity | Slow on large datasets; requires scaling; hard to interpret |

### Key Takeaways
1. All five models achieve **high accuracy (≥ 93%)** on this dataset — it is a well-structured, clean classification problem.
2. Errors are **almost exclusively between versicolor and virginica** — setosa is always classified perfectly.
3. **Random Forest and SVM** tend to be the most robust across different train/test splits.
4. **Decision Tree** is the best choice when interpretability and rule extraction are priorities.
5. **Feature importance** unanimously identifies `petal_length` as the single most informative feature.

---
*Notebook built with Seaborn · Matplotlib · Scikit-learn · Pandas · NumPy*
